# Phenotypic covariance against kinship (GRM, PC-Relate, Weir-Goudet)
Per phenotype and estimator: least-squares line of y_i y_j on kinship over all pairs (slope, intercept, r²), and pairs binned by kinship. Bins with fewer than 21 pairs are not shown.

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
STAGE = "kin_pheno_cov"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
ids = pd.read_csv(target_items(cfg, "grm_shards")[0][0].split(".grm.bin.")[0] + ".grm.id", sep="\t", header=None, dtype=str)[1]
n = len(ids)
est = {"GRM / 2": np.concatenate([np.fromfile(u, dtype="float32") for u, _ in target_items(cfg, "grm_shards")]) / 2}
for lab, st in [("PC-Relate", "pcrelate"), ("Weir-Goudet", "weir_goudet")]:
    u = single_target(cfg, st)
    if os.path.exists(u):
        assert list(pd.read_csv(u.replace(".bin", ".id"), header=None, dtype=str)[0]) == list(ids), st
        est[lab] = np.fromfile(u, dtype="float32")
print(n, "people;", ", ".join(est))
i, j = np.tril_indices(n, -1)
off = np.ones(n * (n + 1) // 2, bool); off[np.arange(1, n + 1) * (np.arange(1, n + 1) + 1) // 2 - 1] = False
est = {k: v[off] for k, v in est.items()}; del off
EDGES = np.unique(np.round(np.r_[np.arange(-0.05, 0.06, 0.005), np.arange(0.06, 0.2, 0.02), [0.2, 0.3, 0.4, 0.6]], 4))
UNREL = 0.05   # "unrelated" fit: pairs with kinship below this

In [ ]:
def fit(k, c):
    """Least squares c = a + b k: intercept, slope, r2."""
    k = k.astype("float64"); km, cm = k.mean(), c.mean()
    sk, sc = k - km, c - cm
    b = (sk @ sc) / (sk @ sk)
    r = (sk @ sc) / np.sqrt((sk @ sk) * (sc @ sc))
    return cm - b * km, b, r * r

rows, binned = [], {}
for uri, item in target_items(cfg, "phenotypes"):
    y = pd.read_csv(uri, sep=" ", dtype={"IID": str}).set_index("IID")["Y"].reindex(ids).to_numpy()
    have = ~np.isnan(y[i]) & ~np.isnan(y[j])
    c = (y[i] * y[j])[have]
    for lab, k in est.items():
        k = k[have]
        for scope, m in [("all pairs", slice(None)), (f"kinship < {UNREL}", k < UNREL)]:
            a, b, r2 = fit(k[m], c[m])
            rows.append(dict(**item, estimator=lab, pairs=scope, n_people=int((~np.isnan(y)).sum()), n_pairs=int(np.size(c[m])),
                             intercept=a, slope=b, r2=r2))
        bi = np.digitize(k, EDGES) - 1; ok = (bi >= 0) & (bi < len(EDGES) - 1)
        cnt = np.bincount(bi[ok], minlength=len(EDGES) - 1)
        binned[(item["pheno"], item["transform"], item["covset"], lab)] = (
            np.bincount(bi[ok], k[ok], len(EDGES) - 1) / np.maximum(cnt, 1), np.bincount(bi[ok], c[ok], len(EDGES) - 1) / np.maximum(cnt, 1), cnt)
    print(os.path.basename(uri), "done")
res = pd.DataFrame(rows)
os.makedirs(os.path.dirname(OUT), exist_ok=True)
with record(cfg, STAGE, OUT) as meta:
    res.to_csv(OUT, sep="\t", index=False)
    meta.update(estimators=list(est), unrelated_cut=UNREL)
rep.table(res.round(6), "estimates", counts=["n_people", "n_pairs"])
pd.set_option("display.width", 200)
display(res[res["pairs"] == "all pairs"].pivot_table(index=["pheno", "transform", "covset"], columns="estimator", values=["slope", "intercept", "r2"]).round(4))

In [ ]:
COV = "base_pcs" if "base_pcs" in cfg.lists["covsets"] else cfg.lists["covsets"][0]
TR = cfg.lists["transforms"][0]
phenos = cfg.lists["phenotypes"]
cols = {"GRM / 2": "#E69F00", "PC-Relate": "#0072B2", "Weir-Goudet": "#009E73"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

def fitrow(ph, lab, scope="all pairs"):
    return res[(res["pheno"] == ph) & (res["transform"] == TR) & (res["covset"] == COV) & (res["estimator"] == lab) & (res["pairs"] == scope)].iloc[0]

NC = min(4, len(phenos)); NR = -(-len(phenos) // NC)
for name, xmax, ylim in [("bulk", 0.05, (-0.04, 0.06)), ("full", 0.6, None)]:
    fig, axes = plt.subplots(NR, NC, figsize=(4.2 * NC, 3.6 * NR), squeeze=False, sharex=True)
    for ax, ph in zip(axes.flat, phenos):
        for lab in est:
            km, cm, cnt = binned[(ph, TR, COV, lab)]
            s = cnt >= 21
            r = fitrow(ph, lab)
            xx = np.array([EDGES[0], xmax])
            ax.plot(xx, r["intercept"] + r["slope"] * xx, color=cols[lab], lw=1.2, alpha=0.6, zorder=1)   # lines behind points
            ax.plot(km[s], cm[s], "o", ms=4.5, color=cols[lab], mec="white", mew=0.5, zorder=2, label=lab)
        ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
        ax.set_xlim(EDGES[0], xmax)
        if ylim:
            ax.set_ylim(*ylim)
        ax.set_title(ph, fontweight="bold"); ax.legend(fontsize=7, frameon=False, loc="upper left")
    for ax in axes.flat[len(phenos):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("kinship")
    for ax in axes[:, 0]:
        ax.set_ylabel("mean $y_i y_j$")
    fig.suptitle(f"{RUN_ID} ({TR}, {COV})", fontsize=11)
    fig.tight_layout()
    rep.fig(fig, f"binned_covariance_{name}", f"{TR}/{COV}; bins with 21+ pairs; lines: least-squares fit over all pairs"); plt.show()

In [ ]:
scopes = list(res["pairs"].unique())
y0 = np.arange(len(phenos))[::-1]
fig, axes = plt.subplots(1, 3, figsize=(15, 0.55 * len(phenos) + 2), sharey=True)
for ax, (v, title) in zip(axes, [("slope", "slope"), ("intercept", "intercept"), ("r2", "r2")]):
    for e, lab in enumerate(est):
        for s, scope in enumerate(scopes):
            d = res[(res["transform"] == TR) & (res["covset"] == COV) & (res["estimator"] == lab) & (res["pairs"] == scope)].set_index("pheno").reindex(phenos)
            ax.plot(d[v], y0 + (1 - e) * 0.25 + (0.5 - s) * 0.08, "o" if s == 0 else "s", ms=6.5, color=cols[lab],
                    mfc=cols[lab] if s == 0 else "white", mew=1.5, ls="none", label=f"{lab}: {scope}")
    if v != "r2":
        ax.axvline(0, color="k", lw=0.6)
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", alpha=0)
for y in y0[::2]:
    for ax in axes:
        ax.axhspan(y - 0.5, y + 0.5, color="0.94", zorder=0)
axes[0].set_yticks(y0, phenos); axes[0].set_ylim(-0.6, len(phenos) - 0.4)
axes[1].ticklabel_format(axis="x", style="sci", scilimits=(-3, 3)); axes[2].set_xscale("log"); axes[2].xaxis.set_minor_formatter(plt.NullFormatter())
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=len(est), fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"{RUN_ID} ({TR}, {COV})", fontsize=11)
fig.tight_layout(rect=(0, 0.08, 1, 1))
rep.fig(fig, "fit_coefficients", f"least-squares fit of y_i y_j on kinship, {TR}/{COV}; filled: all pairs, open: kinship < {UNREL}"); plt.show()
rep.save()